# Pandas from First Principles
## Notebook 6: Applying Functions

---

**Series:** Pandas from First Principles | **Notebook:** 6 of N  
**Topics:** `apply()`, `map()`, `np.where()`, `pd.cut()`, `pd.qcut()`

---

### What you will learn

| # | Topic | Key idea |
|---|-------|----------|
| 1 | Introduction | When to use custom functions vs vectorized ops |
| 2 | `apply()` on a Series | Apply a function to every element |
| 3 | `apply()` on a DataFrame | Apply row-wise or column-wise |
| 4 | `map()` on a Series | Map elements to new values via function/dict |
| 5 | `np.where()` | Vectorized if-else, much faster than apply |
| 6 | `pd.cut()` | Bin continuous data into equal-width categories |
| 7 | `pd.qcut()` | Bin continuous data into equal-frequency categories |

In [ ]:
import pandas as pd
import numpy as np

# Dataset used throughout this notebook
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace'],
    'salary': [85000, 72000, 91000, 68000, 75000, 95000, 64000],
    'age': [28, 34, 26, 41, 30, 35, 27],
    'department': ['Eng', 'Marketing', 'Eng', 'HR', 'Marketing', 'Eng', 'HR'],
    'performance_score': [88, 74, 91, 62, 85, 95, 70],
})

print(employees)

---
## Section 1 — Introduction: Custom Functions vs. Vectorized Operations

### What is it?
Pandas is built on NumPy arrays. Many operations (addition, comparison, string methods) are **vectorized** — they operate on the entire array at C speed without any Python loop.  
Sometimes your transformation logic is too complex for a single vectorized expression. That's when you reach for `apply()` or `map()`.

### The Golden Rule
> ✅ **Always try vectorized operations first.** Use `apply()` only when a vectorized solution does not exist or would be unreadable.

### Why does it matter?
- **Vectorized:** Runs in compiled C/Fortran. Extremely fast.
- **`apply()`:** Runs a Python `for` loop internally. 10×–100× slower on large data.

### Decision tree

```
Need to transform data?
    │
    ├─ Simple arithmetic / comparison?  →  Use vectorized ops  (+, *, >, etc.)
    ├─ Simple if-else on one condition? →  Use np.where()
    ├─ Map each value to another value? →  Use series.map(dict/function)
    ├─ Bin a continuous column?         →  Use pd.cut() / pd.qcut()
    └─ Complex logic across columns?    →  Use DataFrame.apply(axis=1)
```

In [ ]:
# Vectorized vs apply() — speed comparison
import time

# Create a large series for benchmarking
large_series = pd.Series(range(1, 500_001))

# --- Vectorized ---
start = time.time()
result_vectorized = large_series * 2
vectorized_time = time.time() - start

# --- apply() ---
start = time.time()
result_apply = large_series.apply(lambda x: x * 2)
apply_time = time.time() - start

print(f"Vectorized time : {vectorized_time:.4f} seconds")
print(f"apply() time    : {apply_time:.4f} seconds")
print(f"apply() is ~{apply_time / vectorized_time:.0f}x slower for this simple operation")
print()
print("Takeaway: for x*2, use vectorized. Save apply() for complex logic.")

---
## Section 2 — `apply()` on a Series

### What is it?
`Series.apply()` calls a function once for every element in the Series and returns a new Series of results.

### Syntax
```python
series.apply(func, convert_dtype=True, args=(), **kwargs)
```

| Parameter | Description |
|-----------|-------------|
| `func` | Callable — a named function, lambda, or built-in |
| `args` | Tuple of extra positional args passed after the element |
| `**kwargs` | Extra keyword args forwarded to `func` |

### Return value
- A **Series** (same index, new values)

### Best practices
- Prefer a **named function** over a lambda when the logic is more than one expression — it's easier to test and read.
- Use `args=()` to pass extra parameters instead of using a closure — cleaner and avoids subtle scoping bugs.

### Common mistake
```python
# ❌ Forgetting to pass the element — function is called with no arguments
df['salary'].apply(my_func())   # my_func() is called immediately!

# ✅ Pass the function reference, not the call
df['salary'].apply(my_func)
```

In [ ]:
# --- apply() with a named function ---

def salary_category(salary):
    """Classify salary into Low / Medium / High."""
    if salary < 70_000:
        return 'Low'
    elif salary < 90_000:
        return 'Medium'
    else:
        return 'High'

employees['salary_band'] = employees['salary'].apply(salary_category)

print(employees[['name', 'salary', 'salary_band']])

In [ ]:
# --- apply() with a lambda function ---
# Convert salary to thousands (e.g., 85000 → 85.0)

employees['salary_k'] = employees['salary'].apply(lambda x: round(x / 1000, 1))

print(employees[['name', 'salary', 'salary_k']])

In [ ]:
# --- apply() with extra arguments using args=() ---

def apply_bonus(salary, bonus_rate):
    """Return salary after adding a bonus."""
    return salary * (1 + bonus_rate)

# Pass bonus_rate=0.10 via args=()
employees['salary_with_bonus'] = employees['salary'].apply(apply_bonus, args=(0.10,))

print(employees[['name', 'salary', 'salary_with_bonus']])

---
## Section 3 — `apply()` on a DataFrame

### What is it?
`DataFrame.apply()` applies a function along an axis — either across **rows** or across **columns**.

### Syntax
```python
df.apply(func, axis=0, args=(), **kwargs)
```

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `axis` | `0` or `'index'` | Function receives **each column** as a Series |
| `axis` | `1` or `'columns'` | Function receives **each row** as a Series |

### Visual intuition
```
axis=0  (default):          axis=1:
Applies ↓ down each col     Applies → across each row

 A   B   C                   A   B   C
[1] [4] [7]  ← func(col_A)  [1   4   7]  ← func(row_0)
[2] [5] [8]                 [2   5   8]  ← func(row_1)
[3] [6] [9]                 [3   6   9]  ← func(row_2)
```

### Common mistake
```python
# ❌ Forgetting axis=1 when you want row-wise operation
df.apply(lambda row: row['salary'] + row['age'])   # Error or wrong result

# ✅ Correct
df.apply(lambda row: row['salary'] + row['age'], axis=1)
```

In [ ]:
# --- axis=0: function applied to each COLUMN ---
# Find the range (max - min) of each numeric column

numeric_cols = employees[['salary', 'age', 'performance_score']]

col_range = numeric_cols.apply(lambda col: col.max() - col.min(), axis=0)

print("Range (max - min) per column:")
print(col_range)

In [ ]:
# --- axis=1: function applied to each ROW ---
# Compute a composite score = (salary / 1000) * 0.4 + performance_score * 0.6

def composite_score(row):
    """Weighted score combining salary and performance."""
    salary_component = (row['salary'] / 1000) * 0.4
    perf_component = row['performance_score'] * 0.6
    return round(salary_component + perf_component, 2)

employees['composite_score'] = employees.apply(composite_score, axis=1)

print(employees[['name', 'salary', 'performance_score', 'composite_score']])

---
## Section 4 — `map()` on a Series

### What is it?
`Series.map()` transforms each element of a Series using a **function**, a **dictionary**, or another **Series**.

### Syntax
```python
series.map(arg, na_action=None)
```

| `arg` type | Behaviour |
|------------|-----------|
| **dict** | Looks up each value; unmatched keys → `NaN` |
| **function** | Calls function(element) for every element |
| **Series** | Uses the Series' index as the lookup key |

### `map()` vs `apply()` — key differences

| Feature | `map()` | `apply()` |
|---------|---------|----------|
| Works on | Series only | Series **and** DataFrame |
| Accepts dict/Series | ✅ Yes | ❌ No |
| NaN handling | Passes `NaN` to function | Skips `NaN` by default |
| Best for | Element-wise value substitution | Complex transformations |

### Best practices
- Use `map(dict)` for **lookup-table style** substitution — it's the most readable and fast.
- If a key might be missing from the dict, handle it: `map(dict).fillna('Unknown')`.

### Common mistake
```python
# ❌ Using apply() when map(dict) is cleaner
df['dept'].apply(lambda x: {'Eng': 'Engineering'}.get(x, 'Unknown'))

# ✅ map() with a dict is cleaner
df['dept'].map({'Eng': 'Engineering'}).fillna('Unknown')
```

In [ ]:
# --- map() with a dict: expand department abbreviations ---

dept_names = {
    'Eng': 'Engineering',
    'Marketing': 'Marketing & Sales',
    'HR': 'Human Resources',
}

employees['department_full'] = employees['department'].map(dept_names)

print(employees[['name', 'department', 'department_full']])

In [ ]:
# --- map() with a function ---
# Mask names: keep first 2 chars + asterisks for privacy

def mask_name(name):
    return name[:2] + '*' * (len(name) - 2)

masked_names = employees['name'].map(mask_name)

print("Original names:", employees['name'].tolist())
print("Masked  names :", masked_names.tolist())

In [ ]:
# --- NaN handling: map() passes NaN to function, apply() skips it ---

series_with_nan = pd.Series([10, None, 30, None, 50])

print("Series with NaN:")
print(series_with_nan)

# map() passes NaN to the function
map_result = series_with_nan.map(lambda x: f"val={x}")
print("\nmap() result (NaN passed to function):")
print(map_result)

# apply() skips NaN by default
apply_result = series_with_nan.apply(lambda x: f"val={x}")
print("\napply() result (NaN also passed — same here, but behavior differs for complex funcs):")
print(apply_result)

# na_action='ignore' tells map() to skip NaN
map_ignore_nan = series_with_nan.map(lambda x: f"val={x}", na_action='ignore')
print("\nmap(na_action='ignore') — NaN stays NaN:")
print(map_ignore_nan)

---
## Section 5 — `np.where()` — Vectorized If-Else

### What is it?
`np.where()` is NumPy's vectorized ternary operator. It evaluates a condition on the **entire array at once** — no Python loop — making it dramatically faster than `apply()` for conditional logic.

### Syntax
```python
np.where(condition, value_if_true, value_if_false)
```

| Argument | Description |
|----------|-------------|
| `condition` | Boolean array or expression |
| `value_if_true` | Scalar or array — used where condition is `True` |
| `value_if_false` | Scalar or array — used where condition is `False` |

### Return value
- A **NumPy array** (Pandas wraps it into a Series automatically when assigned to a DataFrame column)

### Nested `np.where()` for multiple conditions
```python
# Like: if A → 'X', elif B → 'Y', else → 'Z'
np.where(condition_A, 'X',
    np.where(condition_B, 'Y', 'Z'))
```
> Keep nesting to 2–3 levels max. Use `np.select()` for many conditions.

### When to use `np.where()` vs `apply()`

| Scenario | Best choice |
|----------|-------------|
| One condition, two outcomes | `np.where()` ✅ |
| 2–3 conditions, few outcomes | Nested `np.where()` ✅ |
| Many conditions | `np.select()` ✅ |
| Complex logic with many variables | `apply(axis=1)` ✅ |

In [ ]:
# --- np.where() basic usage ---
# Label employees as 'Senior' (age > 30) or 'Junior'

employees['seniority'] = np.where(employees['age'] > 30, 'Senior', 'Junior')

print(employees[['name', 'age', 'seniority']])

In [ ]:
# --- Nested np.where() for 3-level classification ---
# performance_score: >= 90 → 'Excellent', >= 75 → 'Good', else → 'Needs Improvement'

employees['perf_label'] = np.where(
    employees['performance_score'] >= 90, 'Excellent',
    np.where(
        employees['performance_score'] >= 75, 'Good',
        'Needs Improvement'
    )
)

print(employees[['name', 'performance_score', 'perf_label']])

In [ ]:
# --- Speed comparison: np.where() vs apply() ---

large_df = pd.DataFrame({'salary': np.random.randint(50_000, 150_000, size=500_000)})

# apply()
start = time.time()
large_df['salary'].apply(lambda x: 'High' if x > 85_000 else 'Low')
apply_t = time.time() - start

# np.where()
start = time.time()
np.where(large_df['salary'] > 85_000, 'High', 'Low')
where_t = time.time() - start

print(f"apply() time   : {apply_t:.4f} seconds")
print(f"np.where() time: {where_t:.4f} seconds")
print(f"np.where() is ~{apply_t / where_t:.0f}x faster for simple conditions")

---
## Section 6 — `pd.cut()` — Equal-Width Binning

### What is it?
`pd.cut()` divides a **continuous** numeric column into **equal-width intervals** (buckets). This is useful for converting ages into groups, salaries into bands, scores into grades, etc.

### Syntax
```python
pd.cut(x, bins, right=True, labels=None, include_lowest=False)
```

| Parameter | Description |
|-----------|-------------|
| `x` | The Series or 1D array to bin |
| `bins` | **int** → number of equal-width bins, **list** → explicit bin edges |
| `labels` | List of strings for the bins (length = number of bins) |
| `right=True` | Interval is **(left, right]** — right end is inclusive |
| `right=False` | Interval is **[left, right)** — left end is inclusive |
| `include_lowest` | If `True`, the lowest bin includes the minimum value |

### Return value
- A **Categorical Series** — memory-efficient and sortable

### Best practices
- Always provide `labels` for human-readable output.
- Use a **list for `bins`** when you want unequal-width boundaries (e.g., [0, 18, 65, 120] for age groups).
- Call `.value_counts()` on the result to see how many records fell in each bin.

### Common mistake
```python
# ❌ Value outside bin range becomes NaN silently!
pd.cut([5, 10, 15], bins=[6, 12])  # 5 is below the lowest edge → NaN

# ✅ Ensure bin edges cover all your data
pd.cut([5, 10, 15], bins=[4, 8, 12, 16])
```

In [ ]:
# --- pd.cut() with integer bins (equal-width, auto edges) ---

# Split salary into 3 equal-width bands
employees['salary_cut_auto'] = pd.cut(
    employees['salary'],
    bins=3,
    labels=['Band A', 'Band B', 'Band C']
)

print(employees[['name', 'salary', 'salary_cut_auto']])
print()
print("Counts per band:")
print(employees['salary_cut_auto'].value_counts().sort_index())

In [ ]:
# --- pd.cut() with explicit bin edges (custom boundaries) ---

age_bins = [20, 30, 40, 50]
age_labels = ['20s', '30s', '40s']

employees['age_group'] = pd.cut(
    employees['age'],
    bins=age_bins,
    labels=age_labels,
    right=True       # (20,30] → 30 is in the '20s' bucket
)

print(employees[['name', 'age', 'age_group']])

In [ ]:
# --- right=True vs right=False ---
# Demonstrates which endpoint is inclusive

sample = pd.Series([10, 20, 30, 40])

right_true  = pd.cut(sample, bins=[10, 20, 30, 40], right=True,  labels=['A', 'B', 'C'])
right_false = pd.cut(sample, bins=[10, 20, 30, 40], right=False, labels=['A', 'B', 'C'])

comparison = pd.DataFrame({
    'value': sample,
    'right=True  (a,b]': right_true,
    'right=False [a,b)': right_false,
})

print(comparison)
print()
print("Note: With right=True, the value 10 is NaN (below the open left edge of the first bin).")
print("      Use include_lowest=True to include the minimum value.")

---
## Section 7 — `pd.qcut()` — Quantile-Based Binning

### What is it?
`pd.qcut()` divides data into bins so that **each bin contains the same number of data points** (equal frequency). Unlike `pd.cut()` which creates equal-width bins, `pd.qcut()` creates bins of equal population.

### Syntax
```python
pd.qcut(x, q, labels=None, duplicates='raise')
```

| Parameter | Description |
|-----------|-------------|
| `x` | The Series to bin |
| `q` | **int** → number of quantiles (4 = quartiles), **list** → custom quantile boundaries like [0, 0.25, 0.5, 0.75, 1] |
| `labels` | List of labels for the bins |
| `duplicates` | How to handle duplicate bin edges: `'raise'` (error) or `'drop'` (merge) |

### Return value
- A **Categorical Series** with quantile-based intervals

### `pd.cut()` vs `pd.qcut()` — side by side

| | `pd.cut()` | `pd.qcut()` |
|--|-----------|-------------|
| **Bin width** | Equal width | Varies |
| **Bin frequency** | Varies | Equal |
| **Use case** | Domain-defined ranges (age groups) | Statistical segmentation (quartiles) |
| **Skewed data** | Many values in one bin | Balanced bins |

### Best practices
- Use `qcut` for **rank-based analysis**: top 25%, bottom 25%, etc.
- If you get a `ValueError: Bin edges must be unique`, set `duplicates='drop'` — this happens when many values are identical.

### Common mistake
```python
# ❌ Confusing cut and qcut
# pd.cut(q=4) → does NOT exist
# pd.qcut(bins=3) → does NOT exist

# ✅ Remember: cut uses 'bins', qcut uses 'q'
pd.cut(series, bins=4)
pd.qcut(series, q=4)
```

In [ ]:
# --- pd.qcut() with integer q (equal-frequency quartiles) ---

employees['salary_quartile'] = pd.qcut(
    employees['salary'],
    q=4,
    labels=['Q1 (Bottom)', 'Q2', 'Q3', 'Q4 (Top)']
)

print(employees[['name', 'salary', 'salary_quartile']])
print()
print("Counts per quartile (should be ~equal):")
print(employees['salary_quartile'].value_counts().sort_index())

In [ ]:
# --- pd.qcut() with custom quantile boundaries ---
# Split at median and top 10%

employees['salary_custom_q'] = pd.qcut(
    employees['salary'],
    q=[0, 0.5, 0.9, 1.0],
    labels=['Below Median', 'Mid-High', 'Top 10%']
)

print(employees[['name', 'salary', 'salary_custom_q']])

In [ ]:
# --- cut() vs qcut() side-by-side comparison ---

# Generate skewed data: most values clustered at the low end
np.random.seed(42)
skewed_salaries = pd.Series(np.concatenate([
    np.random.randint(40_000, 60_000, 15),   # 15 low-salary employees
    np.random.randint(60_000, 80_000, 5),    #  5 mid
    np.random.randint(80_000, 120_000, 2),   #  2 high
]))

cut_result  = pd.cut(skewed_salaries,  bins=3, labels=['Low', 'Mid', 'High'])
qcut_result = pd.qcut(skewed_salaries, q=3,    labels=['Bottom 33%', 'Middle 33%', 'Top 33%'])

comparison = pd.DataFrame({
    'salary': skewed_salaries,
    'cut (equal-width)': cut_result,
    'qcut (equal-freq)': qcut_result,
})

print("--- Bin frequency comparison (skewed data) ---")
print("\ncut() counts:")
print(cut_result.value_counts().sort_index())
print("\nqcut() counts:")
print(qcut_result.value_counts().sort_index())
print()
print("Notice: cut() puts most rows in 'Low' because data is skewed.")
print("        qcut() balances the counts across bins.")

---
## Practice Questions

Use the `employees` DataFrame defined at the top of this notebook.

```python
employees = pd.DataFrame({
    'name': ['Alice','Bob','Carol','Dave','Eve','Frank','Grace'],
    'salary': [85000,72000,91000,68000,75000,95000,64000],
    'age': [28,34,26,41,30,35,27],
    'department': ['Eng','Marketing','Eng','HR','Marketing','Eng','HR'],
    'performance_score': [88,74,91,62,85,95,70],
})
```

---

**Q1 [Easy]** — Use `apply()` to add a column `'salary_class'` where:  
- salary < 70000 → `'low'`  
- 70000 ≤ salary < 90000 → `'medium'`  
- salary ≥ 90000 → `'high'`

---

**Q2 [Easy]** — Use `map()` with a dict to add `'dept_full'`:  
- `'Eng'` → `'Engineering'`  
- `'Marketing'` → `'Marketing & Sales'`  
- `'HR'` → `'Human Resources'`

---

**Q3 [Easy]** — Use `np.where()` to add a boolean column `'is_senior'`:  
- `True` if age > 30, otherwise `False`

---

**Q4 [Easy]** — Use `pd.cut()` to add `'salary_band'` with 3 equal-width bins:  
- Labels: `'Entry'`, `'Mid'`, `'Senior'`

---

**Q5 [Medium]** — Use `apply(axis=1)` to add `'combined_score'`:  
- Formula: `(salary / 1000) + performance_score`  
- Round to 2 decimal places

---

**Q6 [Medium]** — Use `pd.qcut()` to split `salary` into 4 quartiles. Add column `'salary_q'`:  
- Labels: `'Q1'`, `'Q2'`, `'Q3'`, `'Q4'`  
- Print the value counts for each quartile

---

**Q7 [Medium]** — Use nested `np.where()` to add `'pay_tier'`:  
- salary > 85000 → `'high'`  
- 70000 ≤ salary ≤ 85000 → `'medium'`  
- salary < 70000 → `'low'`

---

**Q8 [Medium]** — Use `apply()` with `args=()` to add `'after_tax_salary'`:  
- Apply a tax rate of 30% using a named function that accepts `salary` and `tax_rate`  
- Formula: `salary * (1 - tax_rate)`

In [ ]:
# Your answers here — try each question before looking at the answer key!

# Re-create a clean employees DataFrame
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace'],
    'salary': [85000, 72000, 91000, 68000, 75000, 95000, 64000],
    'age': [28, 34, 26, 41, 30, 35, 27],
    'department': ['Eng', 'Marketing', 'Eng', 'HR', 'Marketing', 'Eng', 'HR'],
    'performance_score': [88, 74, 91, 62, 85, 95, 70],
})

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:

# Q6:

# Q7:

# Q8:

---
## Answer Key

In [ ]:
# Clean slate for answers
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace'],
    'salary': [85000, 72000, 91000, 68000, 75000, 95000, 64000],
    'age': [28, 34, 26, 41, 30, 35, 27],
    'department': ['Eng', 'Marketing', 'Eng', 'HR', 'Marketing', 'Eng', 'HR'],
    'performance_score': [88, 74, 91, 62, 85, 95, 70],
})

# -----------------------------------------------------------------------
# Q1 [Easy] — apply() to classify salary
# -----------------------------------------------------------------------
def classify_salary(salary):
    if salary < 70_000:
        return 'low'
    elif salary < 90_000:
        return 'medium'
    else:
        return 'high'

employees['salary_class'] = employees['salary'].apply(classify_salary)
print("Q1 — salary_class:")
print(employees[['name', 'salary', 'salary_class']])
print()

In [ ]:
# -----------------------------------------------------------------------
# Q2 [Easy] — map() with a dict to expand department names
# -----------------------------------------------------------------------
dept_map = {
    'Eng': 'Engineering',
    'Marketing': 'Marketing & Sales',
    'HR': 'Human Resources',
}

employees['dept_full'] = employees['department'].map(dept_map)

print("Q2 — dept_full:")
print(employees[['name', 'department', 'dept_full']])
print()

In [ ]:
# -----------------------------------------------------------------------
# Q3 [Easy] — np.where() to create boolean 'is_senior' column
# -----------------------------------------------------------------------
employees['is_senior'] = np.where(employees['age'] > 30, True, False)

print("Q3 — is_senior:")
print(employees[['name', 'age', 'is_senior']])
print()

In [ ]:
# -----------------------------------------------------------------------
# Q4 [Easy] — pd.cut() into 3 equal-width bands
# -----------------------------------------------------------------------
employees['salary_band'] = pd.cut(
    employees['salary'],
    bins=3,
    labels=['Entry', 'Mid', 'Senior']
)

print("Q4 — salary_band:")
print(employees[['name', 'salary', 'salary_band']])
print("\nCounts:")
print(employees['salary_band'].value_counts().sort_index())
print()

In [ ]:
# -----------------------------------------------------------------------
# Q5 [Medium] — apply(axis=1) combined_score
# -----------------------------------------------------------------------
def combined_score(row):
    return round((row['salary'] / 1000) + row['performance_score'], 2)

employees['combined_score'] = employees.apply(combined_score, axis=1)

print("Q5 — combined_score:")
print(employees[['name', 'salary', 'performance_score', 'combined_score']])
print()

In [ ]:
# -----------------------------------------------------------------------
# Q6 [Medium] — pd.qcut() into 4 salary quartiles
# -----------------------------------------------------------------------
employees['salary_q'] = pd.qcut(
    employees['salary'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print("Q6 — salary_q:")
print(employees[['name', 'salary', 'salary_q']])
print("\nValue counts:")
print(employees['salary_q'].value_counts().sort_index())
print()

In [ ]:
# -----------------------------------------------------------------------
# Q7 [Medium] — nested np.where() for 3-tier pay classification
# -----------------------------------------------------------------------
employees['pay_tier'] = np.where(
    employees['salary'] > 85_000, 'high',
    np.where(
        employees['salary'] >= 70_000, 'medium',
        'low'
    )
)

print("Q7 — pay_tier:")
print(employees[['name', 'salary', 'pay_tier']])
print()

In [ ]:
# -----------------------------------------------------------------------
# Q8 [Medium] — apply() with args=(tax_rate,) to compute after-tax salary
# -----------------------------------------------------------------------
def after_tax(salary, tax_rate):
    """Return salary after deducting tax_rate (e.g., 0.30 for 30%)."""
    return round(salary * (1 - tax_rate), 2)

employees['after_tax_salary'] = employees['salary'].apply(after_tax, args=(0.30,))

print("Q8 — after_tax_salary (30% tax rate):")
print(employees[['name', 'salary', 'after_tax_salary']])

---
## Quick Revision Table

| Function | Input | Axis | Best For | Returns |
|----------|-------|------|----------|---------|
| `series.apply(func)` | Series | element | Complex element-wise logic | Series |
| `df.apply(func, axis=0)` | DataFrame | column | Summary per column | Series |
| `df.apply(func, axis=1)` | DataFrame | row | Logic using multiple columns | Series |
| `series.map(dict)` | Series | element | Value substitution / lookup | Series |
| `series.map(func)` | Series | element | Element-wise transformation | Series |
| `np.where(cond, a, b)` | Array/Series | element | Fast vectorized if-else | ndarray |
| `pd.cut(x, bins)` | Series | element | Equal-width binning | Categorical |
| `pd.qcut(x, q)` | Series | element | Equal-frequency binning | Categorical |

---

### When to use what — decision flowchart

```
Transform a column?
 │
 ├─ Simple math (+, *, /, **)?              → Vectorized arithmetic
 ├─ Lookup value from a dictionary?         → series.map(dict)
 ├─ One condition, two outcomes?            → np.where()
 ├─ Two conditions, three outcomes?         → Nested np.where()
 ├─ Many conditions?                        → np.select()
 ├─ Bin into equal-width ranges?            → pd.cut()
 ├─ Bin into equal-frequency quartiles?     → pd.qcut()
 ├─ Complex logic using one column?         → series.apply(func)
 └─ Complex logic using multiple columns?   → df.apply(func, axis=1)
```

---

### Speed ranking (fastest → slowest)
1. 🏆 Vectorized arithmetic (`+`, `*`, `np.log`, etc.)
2. 🥈 `np.where()` — vectorized C loop
3. 🥉 `series.map(dict)` — hash-table lookup
4. 🐢 `series.apply(func)` / `df.apply(func, axis=1)` — Python loop

---
## 3 Interview Questions

---

**IQ1: What is the difference between `apply()` and `map()` on a Pandas Series?**

> **Answer:**  
> Both transform each element of a Series, but they differ in:
> - **Input types:** `map()` accepts a function, a dict, or another Series. `apply()` only accepts a callable.
> - **NaN handling:** `map()` passes `NaN` to the function by default; use `na_action='ignore'` to skip. `apply()` passes `NaN` too but the behavior inside the function differs.
> - **Use case:** `map()` is ideal for value-substitution (dictionary lookups). `apply()` is more powerful and works on both Series and DataFrames with axis control.

---

**IQ2: When would you choose `pd.qcut()` over `pd.cut()`?**

> **Answer:**  
> - **`pd.cut()`** creates bins of **equal width** — all intervals span the same numeric range. This works well when the domain boundaries matter (e.g., standard age groups: 0–18, 18–65, 65+). However, with skewed data, most records fall in one bin.
> - **`pd.qcut()`** creates bins of **equal frequency** — each bin contains roughly the same number of records. This is better for statistical segmentation (e.g., quartiles, percentiles) and ensures balanced groups regardless of data distribution.
> 
> **Rule:** Use `cut()` when the boundaries are domain-defined. Use `qcut()` when you need balanced populations.

---

**IQ3: Why is `apply()` slow, and what are the alternatives?**

> **Answer:**  
> `apply()` runs a Python `for` loop internally — it iterates over each element (or row) in pure Python, losing all the performance benefits of NumPy's vectorized C routines. For a DataFrame with 1 million rows, `apply(axis=1)` can be 50–100× slower than a vectorized equivalent.  
> 
> **Alternatives (in order of preference):**
> 1. **Vectorized arithmetic** — `df['col'] * 2`
> 2. **`np.where()`** — for conditional logic
> 3. **`np.select()`** — for multiple conditions
> 4. **`series.map(dict)`** — for lookup/substitution
> 5. **`pd.cut()` / `pd.qcut()`** — for binning
> 6. **`apply()`** — last resort for genuinely complex logic

---
## What's Next?

### Notebook 7 — GroupBy & Aggregation

You've learned how to transform individual rows and elements. Now it's time to summarize data **by group**:

| Topic | Preview |
|-------|--------|
| `groupby()` basics | Split data into groups by one or more columns |
| Aggregate functions | `mean()`, `sum()`, `count()`, `min()`, `max()` |
| `agg()` with multiple functions | Apply different functions to different columns |
| Named aggregations | Give custom names to aggregated columns |
| `transform()` | Group-aware transformation without losing shape |
| `filter()` | Keep only groups that satisfy a condition |

---

*Happy learning! 🐼*